# Generalized Linear Models for Claim Costs

Generalized linear models (GLMs) are the workhorse of actuarial pricing. They extend ordinary linear regression in two ways:

- The response can follow a distribution other than the normal one, such as **counts** (Poisson) or **positive, right-skewed amounts** (Gamma).
- A **link function** connects the linear predictor to the mean. With a log link, each coefficient multiplies the expected value, which gives **rate ratios** that are easy to explain.

Pricing typically splits total cost into two parts:

- **Frequency:** how many claims a policy generates per year of exposure. We model it with a **Poisson GLM** and an **offset** for exposure.
- **Severity:** the average cost per claim. We model it with a **Gamma GLM** and a log link.

The expected cost per policy-year is **frequency × severity**.

This lesson covers:

1. Simulating a policy portfolio with claims
2. Train/test split
3. The frequency model (Poisson)
4. The severity model (Gamma)
5. Combining them into expected cost
6. Checking calibration and fit
7. Practice exercises, with solutions

We use **statsmodels**, which reports p-values and confidence intervals, like R's `glm()`.

## 1. Setup and simulated portfolio

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(seed=7)
n_policies = 40000

policies = pd.DataFrame({
    "policy_id": np.arange(n_policies),
    "age": rng.integers(18, 80, size=n_policies),
    "region": rng.choice(["urban", "suburban", "rural"], size=n_policies, p=[0.4, 0.35, 0.25]),
    "smoker": rng.choice(["no", "yes"], size=n_policies, p=[0.8, 0.2]),
    # Exposure = fraction of the year the policy was in force
    "exposure": rng.uniform(0.25, 1.0, size=n_policies),
})

# True claim frequency (claims per policy-year), built from the features
log_freq = (
    -2.2
    + 0.012 * (policies["age"] - 40)
    + 0.35 * (policies["smoker"] == "yes")
    + 0.25 * (policies["region"] == "urban")
)
freq = np.exp(log_freq)

# Claim counts follow a Poisson distribution, scaled by exposure
policies["claims"] = rng.poisson(freq * policies["exposure"])

policies.head()

,policy_id,age,region,smoker,exposure,claims
0,0,76,suburban,yes,0.770853,0
1,1,56,rural,no,0.662791,0
2,2,60,urban,no,0.892638,0
3,3,73,suburban,no,0.880190,0
4,4,53,urban,no,0.718045,0


In [2]:
print("policies:", len(policies))
print("total claims:", policies["claims"].sum())
print("claims per policy-year:", round(policies["claims"].sum() / policies["exposure"].sum(), 3))

policies: 40000
total claims: 3781
claims per policy-year: 0.151


Next we simulate the **cost of each claim**. Individual claim costs are positive and right-skewed, which fits a Gamma distribution. Severity rises with smoking status and age.

We create one row per claim, and attach the policy's features to each row.

In [3]:
claims_list = []
for idx, row in policies[policies["claims"] > 0].iterrows():
    for _ in range(row["claims"]):
        claims_list.append(idx)

claims = policies.loc[claims_list].reset_index(drop=True)

mean_severity = np.exp(
    8.0
    + 0.004 * (claims["age"] - 40)
    + 0.30 * (claims["smoker"] == "yes")
)
shape = 2.0
claims["cost"] = rng.gamma(shape=shape, scale=mean_severity / shape)

print("claims rows:", len(claims))
claims[["age", "smoker", "region", "cost"]].head()

claims rows: 3781


,age,smoker,region,cost
0,69,yes,suburban,975.377810
1,36,no,rural,947.405545
2,79,no,suburban,1618.253357
3,68,no,urban,6617.737898
4,78,no,rural,1495.549227


In [4]:
print(claims["cost"].describe().round(0))

count     3781.0
mean      3449.0
std       2510.0
min         54.0
25%       1627.0
50%       2853.0
75%       4712.0
max      19567.0
Name: cost, dtype: float64


## 2. Train/test split

We split **by policy**, not by claim. If one policy's claims ended up in both sets, the model would effectively see the test data during training.

In [5]:
from sklearn.model_selection import train_test_split

train_policies, test_policies = train_test_split(
    policies, test_size=0.25, random_state=7
)

train_claims = claims[claims["policy_id"].isin(train_policies["policy_id"])]
test_claims = claims[claims["policy_id"].isin(test_policies["policy_id"])]

print("train policies:", len(train_policies), " test policies:", len(test_policies))
print("train claims:  ", len(train_claims), " test claims:  ", len(test_claims))

train policies: 30000  test policies: 10000
train claims:   2842  test claims:   939


## 3. Frequency model: Poisson GLM

We model the **number of claims** for each policy. Two details matter:

- **Family:** Poisson, because counts are non-negative integers.
- **Offset:** `log(exposure)`. A policy in force for half a year should have about half the expected claims. An offset adds this to the linear predictor without estimating a coefficient for it.

In [ ]:
freq_model = smf.glm(
    "claims ~ age + C(smoker) + C(region)",
    data=train_policies,
    family=sm.families.Poisson(),
    offset=np.log(train_policies["exposure"]),
).fit()

print(freq_model.summary())

Read the coefficients on the **log scale**. To get rate ratios, exponentiate them. A rate ratio of 1.40 means 40% more claims, all else equal. The table below shows each rate ratio with its 95% confidence interval:

In [ ]:
rate_ratios = pd.DataFrame({
    "rate_ratio": np.exp(freq_model.params),
    "ci_low": np.exp(freq_model.conf_int()[0]),
    "ci_high": np.exp(freq_model.conf_int()[1]),
    "p_value": freq_model.pvalues,
}).round(3)

rate_ratios

Each row is a **multiplicative** effect. For example, the smoker rate ratio shows how much more often smokers claim, holding age and region constant. Note that the `age` row is the effect of **one extra year** of age.

Check that the Poisson model isn't overdispersed. The dispersion ratio (Pearson χ² divided by the residual degrees of freedom) should be near 1. A value well above 1 suggests the variance is larger than the Poisson assumes, and you may need a negative binomial model.

In [ ]:
dispersion = freq_model.pearson_chi2 / freq_model.df_resid
print("dispersion ratio:", round(dispersion, 3))

## 4. Severity model: Gamma GLM

Now we model the **average cost per claim**. We fit only on claims, not on policies with zero claims. Gamma is a good fit for positive, right-skewed costs. Using a **log link** keeps predictions positive and makes coefficients easy to read as cost ratios.

In [ ]:
sev_model = smf.glm(
    "cost ~ age + C(smoker) + C(region)",
    data=train_claims,
    family=sm.families.Gamma(link=sm.families.links.Log()),
).fit()

print(sev_model.summary())

In [ ]:
severity_ratios = pd.DataFrame({
    "cost_ratio": np.exp(sev_model.params),
    "ci_low": np.exp(sev_model.conf_int()[0]),
    "ci_high": np.exp(sev_model.conf_int()[1]),
    "p_value": sev_model.pvalues,
}).round(3)

severity_ratios

## 5. Combining the models: expected cost

Expected cost for a policy is:

> expected annual cost = expected claims per policy-year × expected cost per claim × exposure

Frequency and severity are fitted separately, then multiplied. This is a common and transparent actuarial structure.

In [ ]:
def expected_cost(policies_df, freq_fit, sev_fit):
    expected_claims = freq_fit.predict(policies_df, offset=np.log(policies_df["exposure"]))
    expected_severity = sev_fit.predict(policies_df)
    return expected_claims * expected_severity

test_policies = test_policies.copy()
test_policies["pred_cost"] = expected_cost(test_policies, freq_model, sev_model)
test_policies[["age", "smoker", "region", "exposure", "pred_cost"]].head()

Note: `freq_model.predict` needs the offset passed in explicitly, since the offset isn't a column of `policies_df`. A function such as `expected_cost` keeps this step in one place.

## 6. Checking calibration

A good pricing model is **well calibrated**: over a whole portfolio, predicted total cost should match actual total cost. The **actual-to-expected (A/E) ratio** is the usual check. A value near 1.0 means the model is well calibrated overall.

Actual cost per policy is the sum of the claim costs for that policy. Policies with no claims cost zero.

In [ ]:
actual_by_policy = test_claims.groupby("policy_id")["cost"].sum()

test_policies["actual_cost"] = actual_by_policy.reindex(test_policies.index).fillna(0.0)

total_actual = test_policies["actual_cost"].sum()
total_expected = test_policies["pred_cost"].sum()

print(f"Actual total:   {total_actual:,.0f}")
print(f"Expected total: {total_expected:,.0f}")
print(f"A/E ratio:      {total_actual / total_expected:.3f}")

Calibration can look good overall while hiding problems in groups. Check A/E by segment, such as region or smoker status:

In [ ]:
segment_ae = test_policies.groupby("region").agg(
    actual=("actual_cost", "sum"),
    expected=("pred_cost", "sum"),
)
segment_ae["A/E"] = (segment_ae["actual"] / segment_ae["expected"]).round(3)
segment_ae

A/E close to 1 in every segment is what we want. If one segment is far off, the model is missing something about that group.

## 7. Practice exercises

Try each exercise before you look at the solution. Cells marked `# TODO` are for you to fill in.

**Exercise 1.** Which policies carry the most expected cost? Fit a model with `age` as a categorical variable (`C(age)`) instead of a linear term, and compare its AIC to the linear model's AIC. Which one fits better?

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
print("linear age AIC:", round(freq_model.aic, 1))

cat_age_model = smf.glm(
    "claims ~ C(age) + C(smoker) + C(region)",
    data=train_policies,
    family=sm.families.Poisson(),
    offset=np.log(train_policies["exposure"]),
).fit()
print("categorical age AIC:", round(cat_age_model.aic, 1))
```

A lower AIC means a better fit after penalizing for extra parameters. The categorical age model has many more parameters, so check that the AIC difference justifies the extra complexity.

</details>

**Exercise 2.** Refit the frequency model without the exposure offset. How do the coefficients change? Why is the offset important for policies that aren't in force all year?

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
no_offset = smf.glm(
    "claims ~ age + C(smoker) + C(region)",
    data=train_policies,
    family=sm.families.Poisson(),
).fit()

print(pd.DataFrame({"with_offset": freq_model.params, "no_offset": no_offset.params}).round(3))
```

Without the offset, a policy in force for three months looks like one in force for a full year, so the model understates how often the policy claims per year of exposure. The intercept shifts to compensate, and the other coefficients can change too.

</details>

**Exercise 3.** Compute the A/E ratio separately for smokers and non-smokers. Is the model calibrated for both groups?

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
test_policies.groupby("smoker").agg(
    actual=("actual_cost", "sum"),
    expected=("pred_cost", "sum"),
).assign(ae=lambda d: (d["actual"] / d["expected"]).round(3))
```

Compare each A/E to 1.0. Gaps in one group suggest that the model misses an effect for that group, or that random variation is larger than expected in a smaller group.

</details>

## Summary

- **GLMs** let the response follow a non-normal distribution, with a link function.
- **Poisson** models counts, and an **offset** for exposure makes rates comparable across policies.
- **Gamma** with a log link models positive, skewed costs.
- **Rate ratios** (exponentiated coefficients) make effects easy to explain.
- **Frequency × severity** gives expected cost.
- **A/E ratios**, overall and by segment, check calibration.
- Check **dispersion** and **AIC** to compare and validate models.